# LSTM from scratch

In [26]:
import os
import re
import numpy as np
from collections import defaultdict, Counter

In [44]:
from nltk.corpus import gutenberg

In [45]:
gutenberg.fileids()

LookupError: 
**********************************************************************
  Resource 'gutenberg' not found.
  Please use the NLTK Downloader to obtain the resource:

  >>> import nltk
  >>> nltk.download('gutenberg')

  For more information see: https://www.nltk.org/data.html

  Attempted to load 'corpora/gutenberg'

  Searched in:
    - 'C:\\Users\\108pa/nltk_data'
    - 'c:\\Users\\108pa\\miniconda3\\envs\\py3.10\\nltk_data'
    - 'c:\\Users\\108pa\\miniconda3\\envs\\py3.10\\share\\nltk_data'
    - 'c:\\Users\\108pa\\miniconda3\\envs\\py3.10\\lib\\nltk_data'
    - 'C:\\Users\\108pa\\AppData\\Roaming\\nltk_data'
    - 'C:\\nltk_data'
    - 'D:\\nltk_data'
    - 'E:\\nltk_data'
**********************************************************************


LSTM Flow

- BPE Tokenizer - tokenize into words/subwords
- LSTMCell      - single-step forward + backward
- LSTM          - unroll the cell over the sequence
- Train         - character / subword language-model training loop

In [28]:
class BPETokenizer:
    """
    Minimal Byte-Pair Encoding tokeniser.
 
    Training
    --------
    Start with every character as its own token.
    Repeatedly find the most frequent adjacent pair and merge it into a new
    token.  Repeat for `num_merges` steps.
 
    The merge list fully defines the tokeniser; encode() replays the merges
    in order on unseen text.
    """
 
    def __init__(self, num_merges: int = 50):
        self.num_merges = num_merges
        self.merges: list[tuple[str, str]] = []   # ordered merge rules
        self.vocab: dict[str, int] = {}            # token → id
        self.id_to_token: dict[int, str] = {}
 
    # ── helpers ──────────────────────────────
 
    @staticmethod
    def _word_to_chars(word: str) -> list[str]:
        """Split a word into characters; append </w> end-of-word marker."""
        return list(word) + ["</w>"]
 
    @staticmethod
    def _get_pairs(vocab: dict[tuple, int]) -> Counter:
        """Count all adjacent symbol pairs across the vocabulary."""
        pairs: Counter = Counter()
        for symbols, freq in vocab.items():
            for i in range(len(symbols) - 1):
                pairs[(symbols[i], symbols[i + 1])] += freq
        return pairs
 
    @staticmethod
    def _merge_pair(pair: tuple[str, str], vocab: dict[tuple, int]) -> dict[tuple, int]:
        """Replace every occurrence of `pair` with the merged token."""
        new_vocab: dict[tuple, int] = {}
        bigram = " ".join(pair)                    # e.g. "a b"
        replacement = "".join(pair)                # e.g. "ab"
        for symbols, freq in vocab.items():
            # re-join to string, replace, split back
            joined = " ".join(symbols)
            merged = joined.replace(bigram, replacement)
            new_vocab[tuple(merged.split())] = freq
        return new_vocab
 
    # ── public API ───────────────────────────
 
    def train(self, text: str) -> None:
        """Learn BPE merges from `text`."""
        # build initial vocab: word → frequency, word split into chars
        word_freq: Counter = Counter(re.findall(r'\S+', text.lower()))
        vocab: dict[tuple, int] = {
            tuple(self._word_to_chars(w)): f
            for w, f in word_freq.items()
        }
 
        # learn merges
        for _ in range(self.num_merges):
            pairs = self._get_pairs(vocab)
            if not pairs:
                break
            best = max(pairs, key=pairs.__getitem__)
            vocab = self._merge_pair(best, vocab)
            self.merges.append(best)
 
        # build final vocabulary from all tokens that appear after merging
        all_tokens: set[str] = set()
        for symbols in vocab:
            all_tokens.update(symbols)
        all_tokens.add("<unk>")
 
        self.vocab = {tok: i for i, tok in enumerate(sorted(all_tokens))}
        self.id_to_token = {i: tok for tok, i in self.vocab.items()}
 
    def _tokenise_word(self, word: str) -> list[str]:
        """Apply learned merges to a single word."""
        symbols = self._word_to_chars(word)
        for pair in self.merges:
            bigram = " ".join(pair)
            replacement = "".join(pair)
            while True:
                joined = " ".join(symbols)
                if bigram not in joined:
                    break
                symbols = joined.replace(bigram, replacement, 1).split()
        return symbols
 
    def encode(self, text: str) -> list[int]:
        """Text → list of token ids."""
        ids: list[int] = []
        for word in re.findall(r'\S+', text.lower()):
            for tok in self._tokenise_word(word):
                ids.append(self.vocab.get(tok, self.vocab["<unk>"]))
        return ids
 
    def decode(self, ids: list[int]) -> str:
        """Token ids → text (removes </w> markers)."""
        tokens = [self.id_to_token.get(i, "<unk>") for i in ids]
        return " ".join("".join(tokens).split("</w>")).strip()
 
    @property
    def vocab_size(self) -> int:
        return len(self.vocab)

In [29]:
def sigmoid(x: np.ndarray) -> np.ndarray:
    return 1.0 / (1.0 + np.exp(-np.clip(x, -30, 30)))

def sigmoid_grad(s: np.ndarray) -> np.ndarray:
    return s * (1.0 - s)

def tanh_grad(t: np.ndarray) -> np.ndarray:
    return 1.0 - t ** 2

class LSTMCell:
    """
    input_size: x_t
    hidden_size: h_t
    W shape: (4.H, H+I)     -> rows: [f, i, o, g]
    b shape: (4.H, )

    Forward pass equations
    ----------------------
        z      = W · [h_{t-1}; x_t] + b        concat then linear
        f_t    = σ(z[0:H])                      forget gate
        i_t    = σ(z[H:2H])                     input gate
        o_t    = σ(z[2H:3H])                    output gate
        g_t    = tanh(z[3H:4H])                 candidate cell
        c_t    = f_t ⊙ c_{t-1} + i_t ⊙ g_t    cell state
        h_t    = o_t ⊙ tanh(c_t)               hidden state
    """

    def __init__(self, input_size: int, hidden_size: int):
        self.I = input_size
        self.H = hidden_size

        k = np.sqrt(1.0 / (input_size + hidden_size))
        self.W = np.random.uniform(-k, k, (4*hidden_size, hidden_size + input_size))
        self.b = np.zeros(4*hidden_size)

        self.dW = np.zeros_like(self.W)
        self.db = np.zeros_like(self.b) 

    
    def forward(self, x: np.ndarray, h_prev: np.ndarray, c_prev: np.ndarray) -> tuple[np.ndarray, np.ndarray, dict]:
        ### returns h_t, c_t, and a cache dict for backward
        H = self.H
        hx = np.concatenate([h_prev, x])
        z = self.W @ hx + self.b 

        f = sigmoid(z[0*H : 1*H])
        i = sigmoid(z[1*H : 2*H])
        o = sigmoid(z[2*H : 3*H])
        g = sigmoid(z[3*H : 4*H])

        c = f * c_prev + i * g # new cell state
        tanh_c = np.tanh(c)
        h = o * tanh_c 

        cache = dict(hx=hx, f=f, i=i, o=o, g=g, c=c, tanh_c=tanh_c, c_prev=c_prev)

        return h, c, cache
    

    def backward(self, dh: np.ndarray, dc: np.ndarray, cache: dict) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
        ### BPTT through a single cell step

        ### returns dx, dh_prev, dc_prev
        ### also accumulates dW and db 

        H = self.H 
        f, i, o, g = cache['f'], cache['i'], cache['o'], cache['g']
        c, tanh_c, c_prev, hx = cache['c'], cache['tanh_c'], cache['c_prev'], cache['hx']

        # ∂L/∂o   — from dh
        do = dh * tanh_c                               # (H,)
        # ∂L/∂tanh(c)
        dtanh_c = dh * o                               # (H,)
        # ∂L/∂c   — from both dtanh_c and incoming dc
        dc_full = dtanh_c * tanh_grad(tanh_c) + dc    # (H,)
 
        # ∂L/∂f, ∂L/∂i, ∂L/∂g, ∂L/∂c_prev
        df     = dc_full * c_prev                      # (H,)
        di     = dc_full * g                           # (H,)
        dg     = dc_full * i                           # (H,)
        dc_prev = dc_full * f                          # (H,) — flows back in time
 
        # ∂L/∂pre-activations (before sigmoid / tanh)
        dz_f = df * sigmoid_grad(f)                    # (H,)
        dz_i = di * sigmoid_grad(i)                    # (H,)
        dz_o = do * sigmoid_grad(o)                    # (H,)
        dz_g = dg * tanh_grad(g)                       # (H,)
 
        dz = np.concatenate([dz_f, dz_i, dz_o, dz_g]) # (4H,)
 
        # accumulate parameter gradients
        self.dW += np.outer(dz, hx)                    # (4H, H+I)
        self.db += dz                                  # (4H,)
 
        # gradient w.r.t. [h_prev; x]
        dhx = self.W.T @ dz                            # (H+I,)
        dh_prev = dhx[:H]
        dx      = dhx[H:]
 
        return dx, dh_prev, dc_prev
 
    def zero_grad(self) -> None:
        self.dW[:] = 0.0
        self.db[:] = 0.0

In [30]:
class LSTM:
    def __init__(self, vocab_size: int, embed_dim: int, hidden_size: int):
        self.V = vocab_size
        self.E = embed_dim
        self.H = hidden_size

        # embedding
        self.embed = np.random.randn(vocab_size, embed_dim) * 0.01

        # LSTM cell
        self.cell = LSTMCell(embed_dim, hidden_size)

        # output projection
        self.W_out = np.random.randn(vocab_size, hidden_size) * 0.01
        self.b_out = np.zeros(vocab_size)

        # gradient accumulators for embedding and output layerr
        self.d_embed = np.zeros_like(self.embed)
        self.dW_out  = np.zeros_like(self.W_out)
        self.db_out  = np.zeros_like(self.b_out)


    def _softmax(self, logits: np.ndarray) -> np.ndarray:
        e = np.exp(logits - logits.max())
        return e / e.sum()
    

    def forward(
            self,
            token_ids: list[int],
            h0: np.ndarray | None = None,
            c0: np.ndarray | None = None,
    ) -> tuple[list[np.ndarray], list[np.ndarray], list[dict], list[np.ndarray]]:
        
        """
        Forward pass over a full sequence

        Returns
        ______
        hs      : list of h_t for each step (T * H)
        cs      : list of c_t for each step (T * H)
        caches  : list of per-step caches   (for backward)
        logits  : list of raw output logits (T * V)
        """

        T = len(token_ids)
        h = np.zeros(self.H) if h0 is None else h0
        c = np.zeros(self.H) if c0 is None else c0 

        hs, cs, caches, logits = [], [], [], []
        self._inputs = token_ids 

        for t in range(T):
            x = self.embed[token_ids[t]]
            h, c, cache = self.cell.forward(x, h, c)
            logit = self.W_out @ h + self.b_out 
            hs.append(h)
            cs.append(c)
            caches.append(cache)
            logits.append(logit)

        return hs, cs, caches, logits
    

    def cross_entropy_loss(self, logits: list[np.ndarray], targets: list[int]) -> tuple[float, list[np.ndarray]]:
        ### compute mean cross-entropy loss and per-step gradient

        T = len(targets)
        loss = 0.0
        d_logits = []
        for t in range(T):
            probs = self._softmax(logits[t])
            loss -= np.log(probs[targets[t]] + 1e-9)

            dl = probs.copy()
            dl[targets[t]] -= 1.0
            d_logits.append(dl / T) # divide by T for mean loss

        return loss / T, d_logits
    

    def backward(self, hs: list[np.ndarray], caches: list[dict], d_logits: list[np.ndarray]) -> None:
        T = len(hs)
        dh_next = np.zeros(self.H)
        dc_next = np.zeros(self.H)

        self.cell.zero_grad()
        self.dW_out[:] = 0.0
        self.db_out[:] = 0.0
        self.d_embed[:] = 0.0

        for t in reversed(range(T)):
            dl = d_logits[t] # get the change in gradident

            # output projection  gradients
            self.dW_out += np.outer(dl, hs[t])
            self.db_out += dl 

            # gradient flowing into h_t from output layer
            dh = self.W_out.T @ dl + dh_next 
            
            dx, dh_next, dc_next = self.cell.backward(dh, dc_next, caches[t])

            # embedding gradient
            self.d_embed[self._inputs[t]] += dx


    def update(self, lr: float = 1e-3, clip: float = 5.0) -> None:
        ### SGD with gradient clipping 
        for param, grad in [
            (self.cell.W,   self.cell.dW),
            (self.cell.b,   self.cell.db),
            (self.W_out,    self.dW_out),
            (self.b_out,    self.db_out),
            (self.embed,    self.d_embed),
        ]:
            np.clip(grad, -clip, clip, out=grad)
            param -= lr * grad


    # ── sampling ─────────────────────────────

    def sample(
        self,
        seed_ids: list[int],
        length: int = 40,
        temperature: float = 1.0,
    ) -> list[int]:
        """Autoregressive sampling from the model."""
        h = np.zeros(self.H)
        c = np.zeros(self.H)
        generated = list(seed_ids)
 
        # warm up on seed
        for tid in seed_ids:
            x = self.embed[tid]
            h, c, _ = self.cell.forward(x, h, c)
 
        # sample
        for _ in range(length):
            logit = self.W_out @ h + self.b_out
            probs = self._softmax(logit / temperature)
            next_id = int(np.random.choice(len(probs), p=probs))
            generated.append(next_id)
            x = self.embed[next_id]
            h, c, _ = self.cell.forward(x, h, c)
 
        return generated
 



In [34]:
def train(
        text: str,
        num_merges: int = 80,
        embed_dim: int = 32,
        hidden_size: int = 64,
        seq_len: int = 20,
        epochs: int = 200,
        lr: float = 5e-3):
    
    ### tokenizer
    print("\n[1] Training BPE Tokenizer")
    tok = BPETokenizer(num_merges = num_merges)
    tok.train(text)
    ids = tok.encode(text)
    V   = tok.vocab_size

    print(f"    vocab size : {V}")
    print(f"    tokens     : {len(ids)}")
    print(f"    merges     : {num_merges}")
    print(f"    first 10 tokens: {[tok.id_to_token[i] for i in ids[:10]]}")
 
    if len(ids) < seq_len + 1:
        raise ValueError("Text too short for the given seq_len.")
    

    ### model 
    print("\n[2] Building LSTM model")
    model = LSTM(V, embed_dim, hidden_size)
    total_params = (
        model.embed.size + model.cell.W.size + model.cell.b.size + model.W_out.size + model.b_out.size
    )

    print(f"\ttotal parameters: {total_params}")


    ### training loop
    losses = []
    for epoch in range(1, epochs + 1):
        start = np.random.randint(0, len(ids) - seq_len - 1) # random weights 
        inputs = ids[start: start+seq_len]
        targets = ids[start+1: start+seq_len+1]

        
        hs, cs, caches, logits = model.forward(inputs)
        loss, d_logits = model.cross_entropy_loss(logits, targets)
        model.backward(hs, caches, d_logits)
        model.update(lr=lr)
 
        losses.append(loss)
        if epoch % 20 == 0 or epoch == 1:
            print(f"  epoch {epoch:4d}  loss={loss:.4f}  perplexity={np.exp(loss):.2f}")
        
    # ── sample ───────────────────────────────
    print("\n[4] Sampling from the trained model …\n")
    seed = ids[:3]
    seed_text = tok.decode(seed)
    sampled = model.sample(seed, length=50, temperature=0.8)
    sampled_text = tok.decode(sampled)
    print(f"  seed    : {seed_text!r}")
    print(f"  sampled : {sampled_text!r}")
    print("\nDone.")
 
    return model, tok, losses
 



In [35]:
if __name__ == "__main__":
    CORPUS = (
        "the cat sat on the mat the cat ate the rat "
        "the rat ran from the cat the mat is flat the hat is on the cat "
        "a fat cat sat on a mat a bat and a cat and a rat "
        "the cat in the hat sat and that was that "
        "the dog barked at the cat the cat jumped over the wall "
        "the small dog ran through the yard the yard was full of grass "
        "a black cat chased a gray rat through the dark house "
        "the bird sang in the tree while the cat watched quietly "
        "the fish swam in the pond near the tall grass "
        "a boy and a girl played with the dog in the park "
        "the sun shined on the lake and the water looked bright "
        "the farmer fed the cow and the horse in the field "
        "a red ball rolled under the table near the chair "
        "the mouse hid behind the box because the cat was near "
        "the old man walked down the road with his brown dog "
        "the little girl read a book beside the warm fire "
        "a fast rabbit ran across the green hill near the river "
        "the rain fell on the roof while the family slept quietly "
        "the baby laughed at the funny cat wearing a small hat "
        "the teacher wrote words on the board for the students "
        "the king sat on the throne inside the grand castle "
        "the queen wore a golden crown and walked through the hall "
        "the knight rode a horse across the windy plain "
        "a pirate sailed across the sea looking for hidden treasure "
        "the robot moved through the room and looked at the people "
        "a programmer wrote code on the computer late at night "
        "the scientist studied the stars and the planets in space "
        "the doctor helped the sick child in the hospital "
        "the baker made fresh bread early in the morning "
        "the artist painted a colorful picture of the mountains "
        "the train moved quickly through the snowy valley "
        "the airplane flew above the clouds during the storm "
        "a lonely wolf howled in the forest under the moon "
        "the children built a small house from wood and stone "
        "the captain steered the ship through the rough ocean "
        "a clever fox sneaked behind the fence near the farm "
        "the monkey climbed the tree and grabbed a banana "
        "the lion rested near the river after the long hunt "
        "the tiger walked silently through the jungle at night "
        "the student solved a difficult math problem after dinner "
        "the singer performed a beautiful song before the crowd "
        "the chef cooked soup with rice vegetables and chicken "
        "the gardener planted flowers beside the path near the gate "
        "the wind moved the leaves across the empty street "
        "the owl watched the forest from a tall branch "
        "the spaceship traveled beyond the stars into deep space "
        "the engineer fixed the broken machine in the factory "
        "the child drew a happy sun beside a blue river "
        "the puppy slept beside the warm fireplace all night "
        "the horse carried the rider across the muddy road "
        "the dragon guarded the treasure inside the dark cave "
    ) * 4
 
    model, tokeniser, losses = train(
        text=CORPUS,
        num_merges=40,
        embed_dim=16,
        hidden_size=32,
        seq_len=12,
        epochs=300,
        lr=8e-3,
    )


[1] Training BPE Tokenizer
    vocab size : 92
    tokens     : 5824
    merges     : 40
    first 10 tokens: ['the</w>', 'cat</w>', 's', 'at</w>', 'on</w>', 'the</w>', 'm', 'at</w>', 'the</w>', 'cat</w>']

[2] Building LSTM model
	total parameters: 10780
  epoch    1  loss=4.5203  perplexity=91.87
  epoch   20  loss=4.5174  perplexity=91.59
  epoch   40  loss=4.5085  perplexity=90.79
  epoch   60  loss=4.4974  perplexity=89.78
  epoch   80  loss=4.4901  perplexity=89.13
  epoch  100  loss=4.4941  perplexity=89.49
  epoch  120  loss=4.4723  perplexity=87.56
  epoch  140  loss=4.4715  perplexity=87.48
  epoch  160  loss=4.4896  perplexity=89.09
  epoch  180  loss=4.4576  perplexity=86.28
  epoch  200  loss=4.4606  perplexity=86.54
  epoch  220  loss=4.4109  perplexity=82.34
  epoch  240  loss=4.4485  perplexity=85.50
  epoch  260  loss=4.4074  perplexity=82.05
  epoch  280  loss=4.4511  perplexity=85.72
  epoch  300  loss=4.4094  perplexity=82.22

[4] Sampling from the trained model …


In [41]:
# Encode your sentence as the seed
seed_text = "water is so "
seed_ids = tokeniser.encode(seed_text)

# Sample continuations
sampled_ids = model.sample(seed_ids, length=30, temperature=0.8)
print(tokeniser.decode(sampled_ids))

water is so on psovinor ane<unk>g urat  g d orbed s er an i iraran the r rored n
